# Loops and Conditions: Working Through Orders

In this practice you will use `if`, `elif` and `else` to make decisions about orders, and `for` and `while` loops to process many orders at once. You will also use `break` and `continue` to control a loop.

## Background

The operations team wants to flag high-value orders, skip cancelled orders when adding up revenue, and count orders by city and priority. Each of these is a decision repeated over many orders, which is exactly what loops and conditions are for.

## Data Description

All the practices in this module use **Retail Orders.csv**: 200 orders from 2025 placed with a small online retailer based in Ireland. The data is fictional and was created for teaching.

| Column Name | Description |
|---|---|
| order_id | Unique order identifier |
| order_date | Date the order was placed (YYYY-MM-DD) |
| delivery_date | Date the order was delivered; blank if not yet delivered |
| customer_name | Customer's name, as typed into the order system |
| city | Delivery city |
| product | Product ordered |
| category | Product category: Electronics, Office or Home |
| quantity | Number of units ordered |
| unit_price | Price of one unit, in euro |
| priority | Order priority: Urgent, Normal or Minor |
| status | Order status: Delivered, Shipped or Cancelled |

## 1. Load the Dataset

We import pandas and read the CSV file into a DataFrame. Keep **Retail Orders.csv** in the same folder as this notebook.

In [1]:
import pandas as pd

df = pd.read_csv("Retail Orders.csv")
df.head()

,order_id,order_date,delivery_date,customer_name,city,product,category,quantity,unit_price,priority,status
0,ORD-1001,2025-01-11,2025-01-19,Tadhg Farrell,Limerick,Filing Cabinet,Office,5,120.0,Minor,Delivered
1,ORD-1002,2025-01-11,2025-01-15,Liam Kelly,Dublin,Whiteboard,Office,3,45.0,Normal,Delivered
2,ORD-1003,2025-01-11,2025-01-13,Sean Ryan,Cork,A4 Notebook,Office,3,4.5,Normal,Delivered
3,ORD-1004,2025-01-12,2025-01-16,Sinead Ryan,Cork,Ergonomic Chair,Office,5,189.0,Normal,Delivered
4,ORD-1005,2025-01-15,2025-01-24,Tomas Lynch,Galway,A4 Notebook,Office,1,4.5,Minor,Delivered


## 2. Prepare the First Ten Orders

To loop over orders one at a time, we turn the first ten rows into a list of dictionaries. `to_dict("records")` gives one dictionary per row, which is the structure you met in Lesson 3.

In [2]:
orders = df.head(10).to_dict("records")
orders[0]

{'order_id': 'ORD-1001',
 'order_date': '2025-01-11',
 'delivery_date': '2025-01-19',
 'customer_name': 'Tadhg Farrell',
 'city': 'Limerick',
 'product': 'Filing Cabinet',
 'category': 'Office',
 'quantity': 5,
 'unit_price': 120.0,
 'priority': 'Minor',
 'status': 'Delivered'}

## 3. The if Statement

`if` runs its block only when the condition is True. Order ORD-1001 is checked against a €500 threshold.

In [3]:
first = orders[0]
total = first["quantity"] * first["unit_price"]

if total > 500:
    print(first["order_id"], "is a high-value order:", total)

ORD-1001 is a high-value order: 600.0


## 4. if..elif..else

With `elif` and `else` every order falls into exactly one band.

In [4]:
second = orders[1]
total = second["quantity"] * second["unit_price"]

if total > 500:
    print("High value")
elif total >= 100:
    print("Standard")
else:
    print("Small")

Standard


## 5. A for Loop over the Orders

The loop variable `o` takes each order in turn. The same `if..elif..else` block now runs for all ten orders.

In [5]:
for o in orders:
    total = o["quantity"] * o["unit_price"]
    if total > 500:
        band = "High value"
    elif total >= 100:
        band = "Standard"
    else:
        band = "Small"
    print(o["order_id"], total, band)

ORD-1001 600.0 High value
ORD-1002 135.0 Standard
ORD-1003 13.5 Small
ORD-1004 945.0 High value
ORD-1005 4.5 Small
ORD-1006 745.0 High value
ORD-1007 158.0 Standard
ORD-1008 120.0 Standard
ORD-1009 45.0 Small
ORD-1010 4.5 Small


## 6. Skip Orders with continue

Cancelled orders should not count towards revenue. `continue` skips the rest of the loop body and moves on to the next order. Here we run the loop over all 200 orders.

In [6]:
all_orders = df.to_dict("records")
revenue = 0

for o in all_orders:
    if o["status"] == "Cancelled":
        continue
    revenue = revenue + o["quantity"] * o["unit_price"]

print("Revenue excluding cancelled orders:", round(revenue, 2))

Revenue excluding cancelled orders: 24279.05


## 7. Stop Early with break

`break` ends the loop completely. We find the first Urgent order of the year and stop looking.

In [7]:
for o in all_orders:
    if o["priority"] == "Urgent":
        print("First urgent order:", o["order_id"], "placed on", o["order_date"])
        break

First urgent order: ORD-1009 placed on 2025-01-26


## 8. A while Loop

A `while` loop repeats as long as its condition is True. We add orders one at a time until revenue first passes €5,000, and count how many orders that took.

In [8]:
running_total = 0
i = 0

while running_total <= 5000:
    o = all_orders[i]
    running_total = running_total + o["quantity"] * o["unit_price"]
    i = i + 1

print("Orders needed:", i)
print("Running total:", round(running_total, 2))

Orders needed: 30
Running total: 5142.54


## 9. A repeat Loop with while True

The repeat pattern runs its body at least once and checks the stopping condition at the end. Here we print the delivery charge for 1, 2, 3 ... items until it would exceed €15, using a fixed €5 fee plus €2.50 per item.

In [9]:
items = 1
while True:
    charge = 5 + items * 2.5
    print(items, "items:", charge)
    items = items + 1
    if 5 + items * 2.5 > 15:
        break

1 items: 7.5
2 items: 10.0
3 items: 12.5
4 items: 15.0


## 10. Nested Loops

The outer loop goes through the cities and the inner loop goes through the priorities. For each pair we count the matching orders.

In [10]:
cities = ["Dublin", "Cork", "Galway", "Limerick", "Waterford"]
priorities = ["Urgent", "Normal", "Minor"]

for city in cities:
    for p in priorities:
        count = 0
        for o in all_orders:
            if o["city"] == city and o["priority"] == p:
                count = count + 1
        print(city, p, count)

Dublin Urgent 10
Dublin Normal 22
Dublin Minor 13
Cork Urgent 5
Cork Normal 21
Cork Minor 11
Galway Urgent 7
Galway Normal 22
Galway Minor 9
Limerick Urgent 7
Limerick Normal 13
Limerick Minor 18
Waterford Urgent 9
Waterford Normal 21
Waterford Minor 12


## Summary

In this practice you:

- used `if`, `if..else` and `if..elif..else` to put orders into value bands
- looped over orders with `for`
- skipped cancelled orders with `continue` and stopped at the first urgent order with `break`
- used a `while` loop and the `while True` repeat pattern
- nested loops to count orders by city and priority

In Lesson 5 you will use Python's numeric and string functions to calculate and clean order data.